# MedTrack_DV – Hospital Data Cleaning and Transformation

**Milestone 1 | Module 2**

This notebook reads `hospital_raw_data.csv`, checks the source data, removes exact duplicate rows, standardizes column names and selected text fields, converts date/numeric fields, derives `length_of_stay_days`, and exports `hospital_cleaned.csv`.

**Important:** The supplied dataset contains admissions, hospitals, conditions, billing and length-of-stay information. It does not contain a department field, a reliable unique patient ID, bed capacity, staffing or equipment-use data. This notebook does not fabricate those fields or calculate unsupported operational KPIs.

Place this notebook in the same folder as `hospital_raw_data.csv`, or update `INPUT_FILE` below.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

INPUT_FILE = Path("hospital_raw_data.csv")
OUTPUT_FILE = Path("hospital_cleaned.csv")

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Could not find {INPUT_FILE.resolve()}. "
        "Place hospital_raw_data.csv beside this notebook or change INPUT_FILE."
    )

df = pd.read_csv(INPUT_FILE)
print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns)}")
display(df.head())


## 1. Inspect the source data

In [ ]:
print("Dataset shape:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())

print("\\nMissing values by column:")
display(df.isna().sum().rename("missing_values").to_frame())

print("\\nExact duplicate rows:", int(df.duplicated().sum()))


## 2. Standardize column names and remove exact duplicates

In [ ]:
# Convert headings such as 'Date of Admission' to 'date_of_admission'.
df.columns = (
    df.columns.astype(str)
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

rows_before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)

print(f"Rows before duplicate removal: {rows_before:,}")
print(f"Exact duplicates removed: {duplicates_removed:,}")
print(f"Rows after duplicate removal: {len(df):,}")


## 3. Clean text, dates and numeric fields

In [ ]:
# Remove leading/trailing whitespace from text columns without changing patient names.
text_columns = df.select_dtypes(include=["object"]).columns
for col in text_columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Standardize selected category labels. Hospital names are title-cased so
# capitalization variants such as 'Sons and Miller'/'Sons And Miller' group together.
category_columns = [
    "gender", "blood_type", "medical_condition", "hospital",
    "insurance_provider", "admission_type", "medication", "test_results"
]
for col in category_columns:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip().str.title()

# Parse date columns using coercion so invalid dates become missing values.
for col in ["date_of_admission", "discharge_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Convert numeric columns safely.
for col in ["age", "billing_amount", "room_number"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Derive length of stay in whole days where both dates are valid.
df["length_of_stay_days"] = (
    df["discharge_date"] - df["date_of_admission"]
).dt.days

# Negative stays are invalid; retain the row but mark the derived value missing.
df.loc[df["length_of_stay_days"] < 0, "length_of_stay_days"] = np.nan

display(df.head())


## 4. Validate the cleaned dataset

In [ ]:
print("Cleaned shape:", df.shape)
print("\\nMissing values after cleaning:")
display(df.isna().sum().rename("missing_values").to_frame())

print("\\nDuplicate rows remaining:", int(df.duplicated().sum()))
print("\\nDate range:")
print("Admission:", df["date_of_admission"].min(), "to", df["date_of_admission"].max())
print("Discharge:", df["discharge_date"].min(), "to", df["discharge_date"].max())

print("\\nLength-of-stay summary:")
display(df["length_of_stay_days"].describe())


## 5. Export the Tableau-ready cleaned dataset

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved to: {OUTPUT_FILE.resolve()}")
print(f"Final records: {len(df):,}")
print(f"Final columns: {len(df.columns)}")


## 6. Data limitations

The available source fields support analysis of admissions, admission type, medical condition, hospital, billing, admission/discharge dates and length of stay.

The following project metrics cannot be calculated reliably from this dataset alone:

- Department-level performance (no department field).
- Readmission rate (no stable unique patient identifier and no agreed readmission window).
- Occupancy and bed utilization (no bed capacity or occupied bed-days).
- Staff utilization (no staffing or shift-allocation data).
- Equipment utilization (no equipment inventory or usage records).

Do not estimate these values from patient names, room numbers or hospital names. Obtain trusted operational data before presenting these as measured KPIs.
